# Aula 05 — Documents, metadados e preparação para busca vetorial com LangChain

Esta atividade migra a representação manual de chunks da Aula 04 para o formato padrão `Document` do LangChain. Um `Document` armazena o texto em `page_content` e informações descritivas em `metadata`; o embedding permanece sob responsabilidade da vector store.

O notebook cobre os dois exercícios fornecidos:

1. criação manual de objetos `Document`;
2. projeto de um schema de metadados e conversão de um chunk real da Aula 04.


In [1]:
import json
from pathlib import Path

from langchain_core.documents import Document

BASE_DIR = Path.cwd()
if BASE_DIR.name == "AULA_05":
    REPO_DIR = BASE_DIR.parent
else:
    REPO_DIR = BASE_DIR

AULA_04_RESULTS = REPO_DIR / "AULA_04" / "results"
assert AULA_04_RESULTS.exists(), "Execute a partir da raiz do repositório ou de AULA_05."


## Exercício 1 — Criando Documents na mão

Foram criados seis documentos curtos sobre quatro temas do curso. Os metadados incluem valores simples, uma lista e um dicionário aninhado para verificar o comportamento da classe.


In [2]:
documentos = [
    Document(
        page_content="Embeddings são representações vetoriais densas de textos.",
        metadata={
            "fonte": "anotacoes_embeddings.md", "pagina": 1,
            "tipo": "teoria", "tema": "embeddings", "autor": "Fernanda Fregulha",
        },
    ),
    Document(
        page_content="Similaridade de cosseno compara a direção de dois vetores.",
        metadata={
            "fonte": "anotacoes_embeddings.md", "pagina": 2,
            "tipo": "exemplo", "tema": "embeddings", "autor": "Fernanda Fregulha",
            "palavras_chave": ["vetores", "cosseno", "similaridade"],
        },
    ),
    Document(
        page_content="Chunking divide documentos longos em unidades menores de contexto.",
        metadata={
            "fonte": "anotacoes_chunking.md", "pagina": 1,
            "tipo": "teoria", "tema": "chunking", "autor": "Fernanda Fregulha",
        },
    ),
    Document(
        page_content="Overlap repete parte do texto para preservar contexto entre chunks.",
        metadata={
            "fonte": "anotacoes_chunking.md", "pagina": 2,
            "tipo": "exemplo", "tema": "chunking", "autor": "Fernanda Fregulha",
            "configuracao": {"chunk_size": 1000, "chunk_overlap": 100},
        },
    ),
    Document(
        page_content="RAG combina recuperação de documentos com geração de respostas.",
        metadata={
            "fonte": "anotacoes_rag.md", "pagina": 1,
            "tipo": "teoria", "tema": "RAG", "autor": "Fernanda Fregulha",
        },
    ),
    Document(
        page_content="Tokenização transforma texto em unidades processadas pelo modelo.",
        metadata={
            "fonte": "anotacoes_tokenizacao.md", "pagina": 1,
            "tipo": "teoria", "tema": "tokenização", "autor": "Fernanda Fregulha",
        },
    ),
]

for indice, documento in enumerate(documentos, start=1):
    print(f"Documento {indice}")
    print("page_content:", documento.page_content)
    print("metadata:", documento.metadata)
    print("-" * 80)

print("len(documentos) =", len(documentos))
assert len(documentos) >= 5


Documento 1
page_content: Embeddings são representações vetoriais densas de textos.
metadata: {'fonte': 'anotacoes_embeddings.md', 'pagina': 1, 'tipo': 'teoria', 'tema': 'embeddings', 'autor': 'Fernanda Fregulha'}
--------------------------------------------------------------------------------
Documento 2
page_content: Similaridade de cosseno compara a direção de dois vetores.
metadata: {'fonte': 'anotacoes_embeddings.md', 'pagina': 2, 'tipo': 'exemplo', 'tema': 'embeddings', 'autor': 'Fernanda Fregulha', 'palavras_chave': ['vetores', 'cosseno', 'similaridade']}
--------------------------------------------------------------------------------
Documento 3
page_content: Chunking divide documentos longos em unidades menores de contexto.
metadata: {'fonte': 'anotacoes_chunking.md', 'pagina': 1, 'tipo': 'teoria', 'tema': 'chunking', 'autor': 'Fernanda Fregulha'}
--------------------------------------------------------------------------------
Documento 4
page_content: Overlap repete parte do 

In [3]:
# Testes solicitados sobre os valores aceitos em metadata.
doc_com_estruturas = Document(
    page_content="Documento para testar estruturas em metadata.",
    metadata={
        "lista": ["embedding", "chunking", "RAG"],
        "dicionario_aninhado": {"modelo": "MiniLM", "dimensao": 384},
        "inteiro": 10,
        "decimal": 0.75,
        "booleano": True,
        "nulo": None,
    },
)
doc_sem_metadata = Document(page_content="Documento criado sem metadata explícito.")

print("Metadata com lista e dicionário:")
print(doc_com_estruturas.metadata)
print("\nMetadata omitido:")
print(doc_sem_metadata.metadata)

assert isinstance(doc_com_estruturas.metadata["lista"], list)
assert isinstance(doc_com_estruturas.metadata["dicionario_aninhado"], dict)
assert doc_sem_metadata.metadata == {}


Metadata com lista e dicionário:
{'lista': ['embedding', 'chunking', 'RAG'], 'dicionario_aninhado': {'modelo': 'MiniLM', 'dimensao': 384}, 'inteiro': 10, 'decimal': 0.75, 'booleano': True, 'nulo': None}

Metadata omitido:
{}


### Respostas do Exercício 1

**Que tipos de dado são aceitos em `metadata`?**  
`metadata` é um dicionário Python e o `Document` aceita valores como texto, números, booleanos, `None`, listas e dicionários aninhados. O teste acima mostra que uma lista e um dicionário são preservados normalmente. Entretanto, algumas vector stores aceitam filtros somente sobre valores escalares; nesse caso, estruturas aninhadas precisam ser convertidas ou achatadas antes da indexação.

**O que acontece sem passar `metadata`?**  
O objeto é criado normalmente e `documento.metadata` recebe um dicionário vazio (`{}`). Apenas `page_content` é obrigatório para este exemplo.


## Exercício 2 — Projetando o schema de metadados

O schema combina os sete campos obrigatórios com campos próprios voltados para citação, inspeção e filtragem.

| Campo | Tipo | Origem | Finalidade |
|---|---|---|---|
| `fonte` | `str` | obrigatório | Nome do Markdown de origem |
| `documento_id` | `str` | obrigatório | Identificador estável do documento |
| `chunk_index` | `int` | obrigatório | Posição do chunk dentro do documento/teste |
| `estrategia` | `str` | obrigatório | Estratégia usada na Aula 04 |
| `chunk_size` | `int` ou `None` | obrigatório | Limite configurado, quando aplicável |
| `chunk_overlap` | `int` | obrigatório | Overlap configurado |
| `n_caracteres` | `int` | obrigatório | Tamanho real do texto |
| `chunk_id` | `str` | próprio | Identificador único para auditoria e deduplicação |
| `secao` | `str` ou `None` | próprio | Heading/seção semântica do trecho |
| `pagina` | `int` ou `None` | próprio | Página original, quando preservada pela extração |
| `n_tokens_estimados` | `int` | próprio | Estimativa de custo e compatibilidade com modelos |
| `caminho_origem` | `str` | próprio | Localizador exato do JSON de onde o chunk veio |
| `idioma` | `str` | próprio | Permite filtrar documentos por idioma |
| `tem_tabela` | `bool` | próprio | Permite localizar chunks com conteúdo tabular |

### Justificativa dos campos próprios

- **`chunk_id`:** qual registro exato originou a resposta e ele já foi indexado?
- **`secao`:** em que seção do documento a informação aparece?
- **`pagina`:** em qual página o usuário pode conferir a informação, quando esse dado existir?
- **`n_tokens_estimados`:** o chunk cabe no contexto do modelo e qual seu custo aproximado?
- **`caminho_origem`:** qual arquivo de resultados deve ser aberto para auditar o chunk?
- **`idioma`:** quais documentos podem responder a uma consulta em determinado idioma?
- **`tem_tabela`:** quais resultados contêm estruturas tabulares que exigem cuidado especial?


In [4]:
def extrair_chunk_index(chunk_id):
    return int(chunk_id.rsplit("chunk", 1)[1])

def encontrar_secao(metadata):
    for nivel in range(6, 0, -1):
        valor = metadata.get(f"heading_{nivel}")
        if valor:
            return valor
    return metadata.get("section") or metadata.get("secao")

def chunk_para_document(chunk, caminho_origem):
    metadata_original = chunk.get("metadata") or {}
    metadata = {
        "fonte": chunk["document_name"],
        "documento_id": chunk["document_id"],
        "chunk_index": extrair_chunk_index(chunk["chunk_id"]),
        "estrategia": chunk["strategy"],
        "chunk_size": chunk["chunk_size"],
        "chunk_overlap": chunk["chunk_overlap"],
        "n_caracteres": chunk["char_count"],
        "chunk_id": chunk["chunk_id"],
        "secao": encontrar_secao(metadata_original),
        "pagina": metadata_original.get("page") or metadata_original.get("pagina"),
        "n_tokens_estimados": chunk["estimated_tokens"],
        "caminho_origem": caminho_origem.as_posix(),
        "idioma": "pt" if chunk["document_id"] in {
            "bioetica_e_ia", "escrita_academica_ia", "twitter_algoritmo"
        } else "en",
        "tem_tabela": "|" in chunk["text"] and "---" in chunk["text"],
    }
    return Document(page_content=chunk["text"], metadata=metadata)

# Exemplo real do teste Markdown, escolhido porque preserva headings nos metadados.
caminho_chunk = AULA_04_RESULTS / "bioetica_e_ia" / "test_10" / "chunks_embeddings.json"
chunks_reais = json.loads(caminho_chunk.read_text(encoding="utf-8"))
chunk_real = next(item for item in chunks_reais if item.get("metadata", {}).get("heading_2"))
documento_real = chunk_para_document(chunk_real, caminho_chunk.relative_to(REPO_DIR))

print("page_content:")
print(documento_real.page_content[:500])
print("\nmetadata:")
print(json.dumps(documento_real.metadata, ensure_ascii=False, indent=2))


page_content:
## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial  
Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1  
1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal.

metadata:
{
  "fonte": "bioetica_e_ia.md",
  "documento_id": "bioetica_e_ia",
  "chunk_index": 2,
  "estrategia": "markdown_headers",
  "chunk_size": null,
  "chunk_overlap": 0,
  "n_caracteres": 227,
  "chunk_id": "bioetica_e_ia_test10_chunk0002",
  "secao": "Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial",
  "pagina": null,
  "n_tokens_estimados": 57,
  "caminho_origem": "AULA_04/results/bioetica_e_ia/test_10/chunks_embeddings.json",
  "idioma": "pt",
  "tem_tabela": false
}


In [5]:
# Exemplo preenchido em JSON. O embedding da Aula 04 não é copiado para Document.
exemplo_json = {
    "page_content": documento_real.page_content,
    "metadata": documento_real.metadata,
}
print(json.dumps(exemplo_json, ensure_ascii=False, indent=2))

assert "embedding" not in documento_real.metadata
assert not hasattr(documento_real, "embedding")


{
  "page_content": "## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial  \nJuracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1  \n1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal.",
  "metadata": {
    "fonte": "bioetica_e_ia.md",
    "documento_id": "bioetica_e_ia",
    "chunk_index": 2,
    "estrategia": "markdown_headers",
    "chunk_size": null,
    "chunk_overlap": 0,
    "n_caracteres": 227,
    "chunk_id": "bioetica_e_ia_test10_chunk0002",
    "secao": "Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial",
    "pagina": null,
    "n_tokens_estimados": 57,
    "caminho_origem": "AULA_04/results/bioetica_e_ia/test_10/chunks_embeddings.json",
    "idioma": "pt",
    "tem_tabela": false
  }
}


### Respostas do Exercício 2

**Qual campo incluir para citar a fonte na resposta do RAG?**  
O campo principal é `caminho_origem`, acompanhado de `fonte`, `pagina`, `secao` e `chunk_id`. Juntos, eles permitem mostrar ao usuário o documento, a localização disponível e o registro exato usado na resposta. Como a conversão da Aula 04 não preservou uma página confiável em todos os chunks, `pagina` aceita `None` e essa limitação não é escondida.

**Por que `chunk_index` é útil?**  
Ele permite recuperar os chunks anterior e seguinte quando o trecho está cortado no meio de uma explicação. Também preserva a ordem, ajuda a reconstruir contexto e facilita depuração e citações.


## Configuração do modelo de embedding

O `Document` não armazena embeddings. A configuração abaixo prepara o modelo local solicitado para uma futura vector store e verifica sua dimensão com uma única frase.


In [6]:
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CACHE_DIR = REPO_DIR / "AULA_04" / ".hf_cache"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    cache_folder=str(CACHE_DIR),
    model_kwargs={"local_files_only": True},
    encode_kwargs={"normalize_embeddings": True},
)

vetor_teste = embeddings.embed_query("Embeddings representam textos como vetores.")
print("Modelo:", EMBEDDING_MODEL)
print("Dimensão:", len(vetor_teste))
print("Primeiros valores:", vetor_teste[:5])
assert len(vetor_teste) == 384


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo: sentence-transformers/all-MiniLM-L6-v2
Dimensão: 384
Primeiros valores: [-0.04111848771572113, 0.010288404300808907, 0.031834784895181656, -0.027147730812430382, -0.01656469888985157]


## Exercício 3 — Busca vetorial com filtros

Para completar o objetivo geral da atividade, os `Document` são indexados em uma `InMemoryVectorStore` do LangChain. A store calcula e administra os embeddings; nenhum vetor é colocado dentro dos documentos.

Para economizar processamento, é usada uma amostra estratificada de até 15 chunks do teste vencedor de cada um dos 12 documentos da Aula 04. Isso mantém todas as fontes representadas, gera no máximo 180 embeddings locais e evita reindexar os 5.026 vetores da atividade anterior.


In [7]:
import numpy as np
from langchain_core.vectorstores import InMemoryVectorStore

MAX_CHUNKS_POR_DOCUMENTO = 15
documentos_indexacao = []

for pasta_documento in sorted(p for p in AULA_04_RESULTS.iterdir() if p.is_dir()):
    caminho = pasta_documento / "test_09" / "chunks_embeddings.json"
    if not caminho.exists():
        continue
    chunks = json.loads(caminho.read_text(encoding="utf-8"))
    quantidade = min(MAX_CHUNKS_POR_DOCUMENTO, len(chunks))
    indices = sorted(set(np.linspace(0, len(chunks) - 1, quantidade, dtype=int).tolist()))
    for indice in indices:
        documentos_indexacao.append(
            chunk_para_document(chunks[indice], caminho.relative_to(REPO_DIR))
        )

print("Documentos representados:", len({d.metadata["documento_id"] for d in documentos_indexacao}))
print("Chunks indexados:", len(documentos_indexacao))
print("Máximo por documento:", MAX_CHUNKS_POR_DOCUMENTO)

vector_store = InMemoryVectorStore(embedding=embeddings)
ids = vector_store.add_documents(
    documents=documentos_indexacao,
    ids=[doc.metadata["chunk_id"] for doc in documentos_indexacao],
)
print("IDs adicionados:", len(ids))

assert len({d.metadata["documento_id"] for d in documentos_indexacao}) == 12
assert len(ids) == len(documentos_indexacao)


Documentos representados: 12
Chunks indexados: 180
Máximo por documento: 15


IDs adicionados: 180


In [8]:
def exibir_resultados(titulo, resultados):
    print("\n" + "=" * 90)
    print(titulo)
    for posicao, (doc, score) in enumerate(resultados, start=1):
        print(f"\n{posicao}. score={score:.4f}")
        print("fonte:", doc.metadata["fonte"])
        print("chunk_id:", doc.metadata["chunk_id"])
        print("caminho:", doc.metadata["caminho_origem"])
        print("trecho:", " ".join(doc.page_content.split())[:300])

consulta_geral = "Como a recuperação de informações melhora respostas em sistemas RAG?"
resultados_gerais = vector_store.similarity_search_with_score(consulta_geral, k=3)
exibir_resultados("Busca sem filtro", resultados_gerais)

resultados_portugues = vector_store.similarity_search_with_score(
    consulta_geral,
    k=3,
    filter=lambda doc: doc.metadata["idioma"] == "pt",
)
exibir_resultados("Busca filtrada por idioma=pt", resultados_portugues)

consulta_atencao = "Como mecanismos de atenção relacionam palavras em uma sequência?"
resultados_atencao = vector_store.similarity_search_with_score(
    consulta_atencao,
    k=3,
    filter=lambda doc: doc.metadata["documento_id"] == "attention_is_all_you_need",
)
exibir_resultados("Busca filtrada por documento", resultados_atencao)

assert all(doc.metadata["idioma"] == "pt" for doc, _ in resultados_portugues)
assert all(
    doc.metadata["documento_id"] == "attention_is_all_you_need"
    for doc, _ in resultados_atencao
)



Busca sem filtro

1. score=0.5776
fonte: twitter_algoritmo.md
chunk_id: twitter_algoritmo_test09_chunk0028
caminho: AULA_04/results/twitter_algoritmo/test_09/chunks_embeddings.json
trecho: Essa transformação da noção de informação é fundamental para compreender como seu impacto reestruturou a ideia sobre espaço público. Com o advento das Novas Tecnologias de Informação e Comunicação, há uma crise entre o processo autoral, que deve ser devida mente regulado e controlado, como forma de 

2. score=0.5725
fonte: escrita_academica_ia.md
chunk_id: escrita_academica_ia_test09_chunk0013
caminho: AULA_04/results/escrita_academica_ia/test_09/chunks_embeddings.json
trecho: | | 3. Apropriação e reescrita autoral | Transformação e personalização do texto | Infundir a voz, pensamento e estilo do autor, ga- rantindo originalidade e evitando marcas algorít- micas. | | Pós- escrita | 4. Verificação rigorosa de fatos e fontes | Validação independente de infor- mações | Asseg

3. score=0.5357
fonte: esc

In [9]:
def serializar_resultados(consulta, filtro, resultados):
    return {
        "consulta": consulta,
        "filtro": filtro,
        "resultados": [
            {
                "posicao": posicao,
                "score": round(float(score), 6),
                "page_content": doc.page_content,
                "metadata": doc.metadata,
            }
            for posicao, (doc, score) in enumerate(resultados, start=1)
        ],
    }

resultado_buscas = {
    "modelo_embedding": EMBEDDING_MODEL,
    "tipo_vector_store": "InMemoryVectorStore",
    "documentos_indexados": len(documentos_indexacao),
    "fontes_representadas": 12,
    "metodologia_economica": (
        "Amostra estratificada de até 15 chunks do teste 09 por documento; "
        "embeddings locais e store em memória."
    ),
    "buscas": [
        serializar_resultados(consulta_geral, None, resultados_gerais),
        serializar_resultados(consulta_geral, {"idioma": "pt"}, resultados_portugues),
        serializar_resultados(
            consulta_atencao,
            {"documento_id": "attention_is_all_you_need"},
            resultados_atencao,
        ),
    ],
}

RESULTS_DIR = REPO_DIR / "AULA_05" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "buscas_exemplo.json").write_text(
    json.dumps(resultado_buscas, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Resultados salvos em:", (RESULTS_DIR / "buscas_exemplo.json").resolve())


Resultados salvos em: C:\Users\Fernanda Fregulha\Downloads\IA\IA\AULA_05\results\buscas_exemplo.json


### Análise das buscas

- A busca sem filtro pode recuperar qualquer uma das 12 fontes.
- O filtro `idioma == "pt"` restringe os candidatos antes do ranking aos três documentos em português.
- O filtro por `documento_id` restringe a busca ao artigo *Attention Is All You Need*.
- Cada resultado mostra `fonte`, `chunk_id` e `caminho_origem`, permitindo citar e auditar a informação.
- `k=3` reduz o volume recuperado e evita enviar contexto desnecessário a um futuro modelo gerador.
- A store é mantida em memória e recriada ao executar o notebook; para esta atividade, isso evita dependências e arquivos de índice adicionais.


## Conclusão

Os chunks da Aula 04 foram migrados para `Document` sem carregar o vetor dentro do objeto. O texto fica em `page_content`, o schema de `metadata` oferece rastreabilidade e filtros, e a `InMemoryVectorStore` calcula os embeddings no momento da indexação. As buscas com filtros confirmam que os metadados permitem restringir fontes e citar os trechos recuperados.
